# Chapter 9: Text Processing and Multiclass Classification

**Referensi:** scikit-learn Cookbook, Third Edition - John Sukup (Packt Publishing, 2025)

---

## Ringkasan Chapter

Chapter 9 membahas Natural Language Processing (NLP) menggunakan scikit-learn, yaitu cara memproses dan mengklasifikasikan data teks. Diperkirakan 80-90% data di dunia tersimpan dalam bentuk tidak terstruktur, dengan teks sebagai salah satu yang paling dominan. Komputer tidak secara native memahami teks, sehingga diperlukan teknik transformasi dari teks mentah menjadi representasi numerik yang dapat diproses oleh algoritma ML.

### Topik yang Dibahas:
1. Pengantar text processing dan preprocessing
2. Teknik vektorisasi teks: Bag-of-Words (CountVectorizer) dan TF-IDF
3. Feature extraction dari teks: n-gram dan POS tagging
4. Implementasi model klasifikasi teks (Naive Bayes, SVM, Logistic Regression)
5. Strategi multiclass classification untuk teks (OvR dan OvO)
6. Evaluasi model teks (classification report, confusion matrix)


---
## Setup: Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources yang diperlukan
import nltk
for resource in ['movie_reviews', 'reuters', 'brown', 'punkt',
                 'punkt_tab', 'averaged_perceptron_tagger_eng',
                 'stopwords']:
    try:
        nltk.data.find(f'corpora/{resource}')
    except LookupError:
        try:
            nltk.data.find(f'tokenizers/{resource}')
        except LookupError:
            nltk.download(resource, quiet=True)

print("Library dan NLTK resources berhasil disetup.")


---
## 1. Pengantar Text Processing

### Penjelasan Teori

**Text processing (pemrosesan teks)** adalah langkah fundamental dalam ML yang melibatkan konversi data teks mentah menjadi representasi numerik yang dapat digunakan oleh algoritma ML. Ini penting karena komputer tidak memahami teks secara native.

**Mengapa text processing penting:**
- Diperkirakan 80-90% data di dunia adalah data tidak terstruktur, termasuk teks.
- Email, artikel berita, review produk, media sosial, dokumen hukum, catatan medis, semuanya adalah teks.
- Dengan teknik NLP yang tepat, semua data ini bisa dianalisis secara otomatis.

**Tantangan dalam text processing:**
1. **Variasi bahasa:** Sinonim, singkatan, typo, bahasa gaul, slang.
2. **Konteks dan ambiguitas:** Kata yang sama bisa memiliki makna berbeda tergantung konteks.
3. **Noise:** HTML tags, tanda baca, angka yang tidak relevan.
4. **Dimensi tinggi:** Setiap kata unik bisa menjadi satu fitur, menghasilkan ribuan dimensi.

**Langkah-langkah umum preprocessing teks:**
1. Lowercase conversion (mengubah semua huruf menjadi huruf kecil)
2. Tokenisasi (memisahkan teks menjadi token/kata individual)
3. Stopword removal (menghapus kata-kata tidak informatif seperti "the", "a", "is")
4. Stemming/Lemmatization (mengembalikan kata ke bentuk dasarnya)
5. Vektorisasi (mengubah token menjadi representasi numerik)

**Token** adalah unit dasar teks dalam NLP: bisa berupa kata, kalimat, paragraf, atau bahkan karakter tunggal, tergantung aplikasinya.

**Corpus** adalah kumpulan dokumen teks yang digunakan dalam konteks pemodelan.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import movie_reviews

# Load movie_reviews corpus dari NLTK
# Dataset berisi 2000 review film yang dikategorikan sebagai positif atau negatif
documents, labels = [], []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        documents.append(movie_reviews.raw(fileid))
        labels.append(category)

# Konversi label ke numerik
label_map = {label: i for i, label in enumerate(sorted(set(labels)))}
numerical_labels = [label_map[l] for l in labels]

print(f"Total dokumen dalam corpus: {len(documents)}")
print(f"Kategori: {sorted(set(labels))} -> {label_map}")
print(f"Distribusi: neg={labels.count('neg')}, pos={labels.count('pos')}")
print()
print("Contoh awal review pertama (100 karakter):")
print(documents[0][:200] + "...")


In [ ]:
# Gunakan subset untuk demonstrasi yang lebih cepat
subset_size = 500
texts_subset, _, labels_subset, _ = train_test_split(
    documents, numerical_labels,
    train_size=subset_size,
    stratify=numerical_labels,
    random_state=2024
)

X_train, X_test, y_train, y_test = train_test_split(
    texts_subset, labels_subset,
    test_size=0.3, random_state=2024,
    stratify=labels_subset
)

print(f"Training set: {len(X_train)} dokumen")
print(f"Test set:     {len(X_test)} dokumen")


In [ ]:
# CountVectorizer: Bag-of-Words dasar dengan stopword removal
vectorizer = CountVectorizer(stop_words='english', max_features=5000)
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect  = vectorizer.transform(X_test)

print(f"Shape sparse matrix (training): {X_train_vect.shape}")
print(f"(Setiap baris = satu dokumen, setiap kolom = satu token unik)")
print(f"Jumlah token unik setelah stopword removal: {len(vectorizer.vocabulary_)}")
print()

# Top 20 token paling sering muncul
feature_names_cv = vectorizer.get_feature_names_out()
counts_cv = X_train_vect.toarray().sum(axis=0)
top20_idx = np.argsort(counts_cv)[-20:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(feature_names_cv[top20_idx][::-1], counts_cv[top20_idx][::-1], color='steelblue')
ax.set_xlabel('Frekuensi Total dalam Training Set')
ax.set_title('Top 20 Token Paling Sering Muncul
(Setelah stopword removal, movie_reviews corpus)')
plt.tight_layout()
plt.show()


---
## 2. Teknik Vektorisasi Teks

### Penjelasan Teori

Vektorisasi teks adalah proses mengubah teks mentah menjadi vektor numerik. Dua pendekatan utama:

**1. Bag-of-Words (BoW) dengan CountVectorizer:**
- Merepresentasikan setiap dokumen sebagai vektor hitungan kemunculan kata.
- Menyederhanakan teks menjadi "kantong kata" tanpa mempertimbangkan urutan atau konteks.
- Sederhana dan efektif untuk banyak tugas klasifikasi teks.
- Kelemahannya: kata yang sering muncul di semua dokumen (misal "the", "film") mendapat bobot tinggi meski tidak informatif.

**2. TF-IDF (Term Frequency-Inverse Document Frequency):**
- Menggabungkan dua konsep:
  - **TF (Term Frequency):** Seberapa sering suatu kata muncul dalam satu dokumen.
  - **IDF (Inverse Document Frequency):** Seberapa jarang suatu kata muncul di seluruh corpus. Kata yang jarang = lebih informatif.
- Formula: `TF-IDF(t, d) = TF(t, d) * log(N / DF(t))`
  di mana N = total dokumen, DF(t) = jumlah dokumen yang mengandung term t.
- Memberikan bobot lebih tinggi kepada kata yang spesifik dan diskriminatif.
- Umumnya menghasilkan performa klasifikasi yang lebih baik dibanding BoW murni.

**Word Embeddings (dense vectors):**
- Seperti Word2Vec dan GloVe, merepresentasikan kata sebagai vektor padat (dense) yang menangkap makna semantik.
- scikit-learn tidak menyediakan implementasi langsung, tapi embedding eksternal bisa diintegrasikan.
- Embedding adalah komponen preprocessing penting dalam LLM modern.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import reuters

# Load Reuters corpus (berita, 90+ kategori topik)
try:
    reuters.fileids()
    documents_r = reuters.fileids()
    texts_r     = [reuters.raw(doc_id) for doc_id in documents_r]
    labels_r    = [reuters.categories(doc_id)[0] for doc_id in documents_r]
    print(f"Reuters corpus: {len(texts_r)} dokumen")
    print(f"Jumlah kategori: {len(set(labels_r))}")
    reuters_loaded = True
except Exception as e:
    print(f"Reuters corpus tidak tersedia: {e}")
    reuters_loaded = False


In [ ]:
# Demonstrasi perbedaan BoW vs TF-IDF pada corpus movie_reviews
# BoW
bow_vec = CountVectorizer(stop_words='english', max_features=3000)
X_bow = bow_vec.fit_transform(X_train)

# TF-IDF
tfidf_vec = TfidfVectorizer(stop_words='english', max_features=3000)
X_tfidf = tfidf_vec.fit_transform(X_train)

print("Perbandingan BoW vs TF-IDF:")
print(f"  BoW shape:   {X_bow.shape} | Rata-rata nilai per cell: {X_bow.mean():.4f}")
print(f"  TF-IDF shape:{X_tfidf.shape} | Rata-rata nilai per cell: {X_tfidf.mean():.4f}")
print()

# Visualisasi: Top 20 token berdasarkan rata-rata TF-IDF weight
feature_names_tfidf = tfidf_vec.get_feature_names_out()
tfidf_means = X_tfidf.toarray().mean(axis=0)
top20_tfidf = np.argsort(tfidf_means)[-20:]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel kiri: BoW
counts_bow = X_bow.toarray().sum(axis=0)
top20_bow  = np.argsort(counts_bow)[-20:]
fn_bow = bow_vec.get_feature_names_out()
axes[0].barh(fn_bow[top20_bow][::-1], counts_bow[top20_bow][::-1], color='steelblue')
axes[0].set_xlabel('Frekuensi Total')
axes[0].set_title('Top 20 Tokens - Bag-of-Words
(berdasarkan frekuensi)')

# Panel kanan: TF-IDF
axes[1].barh(feature_names_tfidf[top20_tfidf][::-1],
             tfidf_means[top20_tfidf][::-1], color='salmon')
axes[1].set_xlabel('Mean TF-IDF Weight')
axes[1].set_title('Top 20 Tokens - TF-IDF
(berdasarkan rata-rata bobot)')

plt.suptitle('Bag-of-Words vs TF-IDF: Token Terpenting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Perhatikan: TF-IDF mengangkat kata-kata diskriminatif yang jarang muncul,")
print("sementara BoW dipengaruhi oleh kata-kata yang sekedar sering muncul.")


---
## 3. Feature Extraction dari Teks

### Penjelasan Teori

Feature extraction dari teks melibatkan pengidentifikasian pola dan atribut yang bermakna dalam data teks untuk meningkatkan performa model klasifikasi.

**Tiga teknik utama:**

**1. N-grams:**
Menangkap urutan N kata berurutan, bukan kata individual. Memberikan konteks yang tidak bisa ditangkap oleh unigram.
- Unigram (N=1): "machine", "learning"
- Bigram (N=2): "machine learning"
- Trigram (N=3): "deep machine learning"

Dalam `CountVectorizer()` dan `TfidfVectorizer()`, gunakan parameter `ngram_range=(min_n, max_n)`.

**2. Part-of-Speech (POS) Tagging:**
Mengklasifikasikan setiap kata berdasarkan kategori gramatikalnya:
- `NN`: Noun (kata benda)
- `VB`: Verb (kata kerja)
- `JJ`: Adjective (kata sifat)
- `RB`: Adverb (kata keterangan)
- `NNP`: Proper Noun (nama orang/tempat)
POS tagging membantu model memahami struktur sintaktis teks, bukan hanya kemunculan kata.

**3. Named Entity Recognition (NER):**
Mengidentifikasi dan mengklasifikasikan entitas bernama seperti nama orang, organisasi, lokasi, tanggal, dll. Berguna untuk analisis teks yang membutuhkan pemahaman entitas spesifik.


In [ ]:
# N-gram extraction dengan CountVectorizer
ngram_vect = CountVectorizer(
    ngram_range=(1, 2),  # Unigram DAN bigram
    stop_words='english',
    max_features=5000
)
X_ngram = ngram_vect.fit_transform(X_train)

print(f"Shape dengan n-gram (1,2): {X_ngram.shape}")
print(f"Contoh bigram yang ditemukan:")
all_features = ngram_vect.get_feature_names_out()
bigrams = [f for f in all_features if ' ' in f][:15]
print("  " + ", ".join(bigrams))


In [ ]:
# POS Tagging demonstration menggunakan NLTK brown corpus
from nltk.corpus import brown
from nltk.util import ngrams as nltk_ngrams
from collections import Counter

# Ambil beberapa kalimat dari brown corpus untuk demonstrasi
sample_texts = []
categories_brown = brown.categories()[:2]  # ambil 2 kategori pertama

for cat in categories_brown:
    sents = brown.sents(categories=cat)[:2]
    for sent in sents:
        sample_texts.append(" ".join(sent))

print("Contoh teks dari Brown corpus:")
for i, t in enumerate(sample_texts[:2]):
    print(f"  [{i+1}] {t}")

print()

# POS Tagging
print("Hasil POS Tagging:")
for i, text in enumerate(sample_texts[:2]):
    print(f"
Kalimat {i+1}: '{text}'")
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    print("  Token/TAG pairs:")
    tag_str = " | ".join([f"{tok}/{tag}" for tok, tag in pos_tags[:10]])
    print(f"  {tag_str}")


In [ ]:
# Distribusi POS tag pada movie_reviews training set
from nltk.corpus import stopwords

stop_words_set = set(stopwords.words('english'))
all_pos_tags   = []

# Sample 50 dokumen untuk kecepatan
for text in X_train[:50]:
    tokens = nltk.word_tokenize(text.lower())
    filtered = [t for t in tokens if t.isalpha() and t not in stop_words_set]
    if filtered:
        tagged = nltk.pos_tag(filtered)
        all_pos_tags.extend([tag for _, tag in tagged])

pos_counter = Counter(all_pos_tags)
top10_pos = pos_counter.most_common(10)
tags, counts = zip(*top10_pos)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(list(tags)[::-1], list(counts)[::-1], color='lightcoral')
ax.set_xlabel('Frekuensi')
ax.set_ylabel('POS Tag')
ax.set_title('Top 10 POS Tag di Movie Reviews Training Set
(Stopwords dihapus)')
plt.tight_layout()
plt.show()

# Legenda POS tags
print("Keterangan POS Tags:")
pos_legend = {
    'NN': 'Noun (singular)', 'NNS': 'Noun (plural)',
    'JJ': 'Adjective', 'VB': 'Verb (base)', 'VBD': 'Verb (past tense)',
    'VBN': 'Verb (past participle)', 'RB': 'Adverb', 'NNP': 'Proper Noun',
    'VBG': 'Verb (gerund)', 'VBZ': 'Verb (3rd person singular)'
}
for tag, desc in pos_legend.items():
    if tag in pos_counter:
        print(f"  {tag}: {desc} (count: {pos_counter[tag]})")


---
## 4. Implementasi Model Klasifikasi Teks

### Penjelasan Teori

Setelah teks dipreprocessing dan divektorisasi, kita dapat menerapkan algoritma ML standar untuk klasifikasi. Tiga algoritma yang paling umum untuk klasifikasi teks:

**1. Naive Bayes (MultinomialNB):**
- Berdasarkan teorema Bayes dengan asumsi independensi antar fitur (naive).
- Sangat efisien untuk data teks karena sparsitas data tidak menjadi masalah.
- Bekerja baik dengan CountVectorizer (hitungan frekuensi).
- Nama "Naive" mengacu pada asumsi independensi yang disederhanakan, yang jarang terpenuhi di dunia nyata, tapi tetap efektif dalam praktik.

**2. SVM (Support Vector Machine):**
- Sangat efektif untuk data teks berdimensi tinggi.
- Menemukan hyperplane optimal yang memisahkan kelas dalam ruang fitur yang sangat besar.
- Bekerja baik dengan TF-IDF karena mengeksploitasi representasi vektor.

**3. Logistic Regression:**
- Memberikan probabilitas yang terkalibrasi dengan baik.
- Interpretable: koefisien menunjukkan kontribusi setiap kata terhadap prediksi kelas.
- Robust dan konsisten untuk berbagai jenis data teks.

Setelah vektorisasi, semua algoritma ini bekerja dengan cara yang sama persis seperti ketika digunakan pada data numerik biasa.


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load brown corpus untuk klasifikasi genre
from nltk.corpus import brown

document_ids  = brown.fileids()
texts_brown   = [' '.join(brown.words(fid)) for fid in document_ids]
raw_labels_br = [brown.categories(fid)[0] for fid in document_ids]

# Encode label ke integer
unique_labels_br = sorted(set(raw_labels_br))
label_to_int_br  = {l: i for i, l in enumerate(unique_labels_br)}
labels_br        = [label_to_int_br[l] for l in raw_labels_br]

X_br_train, X_br_test, y_br_train, y_br_test = train_test_split(
    texts_brown, labels_br,
    test_size=0.3, stratify=labels_br, random_state=2024
)

# Vektorisasi dengan TF-IDF
tfidf_br = TfidfVectorizer(stop_words='english', max_features=5000)
X_br_train_vect = tfidf_br.fit_transform(X_br_train)
X_br_test_vect  = tfidf_br.transform(X_br_test)

print(f"Brown corpus: {len(texts_brown)} dokumen, {len(unique_labels_br)} genre")
print(f"Genre: {unique_labels_br}")
print(f"TF-IDF matrix shape (train): {X_br_train_vect.shape}")


In [ ]:
# Latih dan evaluasi 3 model
models_text = {
    'Naive Bayes':       MultinomialNB(),
    'SVM':               SVC(kernel='linear', random_state=2024),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=2024)
}

text_results = {}
for name, model in models_text.items():
    model.fit(X_br_train_vect, y_br_train)
    y_pred_br = model.predict(X_br_test_vect)
    acc_br    = accuracy_score(y_br_test, y_pred_br)
    text_results[name] = acc_br
    print(f"{name:<25} Accuracy: {acc_br:.4f}")


In [ ]:
# Visualisasi perbandingan akurasi
fig, ax = plt.subplots(figsize=(8, 4))
names_txt = list(text_results.keys())
accs_txt  = list(text_results.values())
colors_txt = ['steelblue', 'salmon', 'green']

bars = ax.bar(names_txt, accs_txt, color=colors_txt, edgecolor='black')
for bar, val in zip(bars, accs_txt):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Accuracy')
ax.set_title('Perbandingan Model Klasifikasi Teks
(Brown Corpus - Genre Classification)')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# Classification report untuk model terbaik (SVM)
best_text_model = models_text['SVM']
y_pred_best_txt = best_text_model.predict(X_br_test_vect)

report_txt = pd.DataFrame(
    classification_report(
        y_br_test, y_pred_best_txt,
        target_names=unique_labels_br,
        output_dict=True, zero_division=0
    )
).transpose()

print("Classification Report - SVM (Brown Corpus, Genre Classification):")
display(report_txt.round(4))


---
## 5. Strategi Multiclass Classification untuk Teks

### Penjelasan Teori

Ketika dataset memiliki lebih dari dua kelas (seperti pada genre klasifikasi), kita perlu strategi multiclass yang tepat. Sebagian besar algoritma ML hanya mendukung binary classification secara native.

**Dua strategi dekomposisi multiclass:**

**1. One-vs-Rest (OvR) - juga disebut One-vs-All:**
- Melatih satu binary classifier per kelas.
- Masing-masing classifier belajar membedakan satu kelas vs semua kelas lainnya.
- Saat prediksi, classifier dengan confidence score tertinggi "menang".
- Lebih efisien secara komputasi (paralel).
- Diimplementasikan dengan `OneVsRestClassifier()`.

**2. One-vs-One (OvO):**
- Melatih satu binary classifier untuk setiap pasangan kelas.
- Untuk K kelas: K*(K-1)/2 classifier.
- Saat prediksi, setiap classifier memberi satu vote, kelas dengan vote terbanyak menang.
- Lebih banyak model, tapi setiap model lebih sederhana (dataset lebih kecil).
- Diimplementasikan dengan `OneVsOneClassifier()`.

**Perbandingan OvR vs OvO untuk klasifikasi teks:**
- OvR: Lebih cepat, cocok untuk dataset besar.
- OvO: Lebih akurat pada dataset seimbang dengan banyak kelas, tapi lebih lambat.

**Logistic Regression** dan **SVM Linear** mendukung multiclass secara native dan sering lebih baik dari OvR/OvO pada praktiknya.


In [ ]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

# Terapkan OvR dan OvO pada Brown corpus
# Vektorisasi dengan TF-IDF
tfidf_mc = TfidfVectorizer(stop_words='english', max_features=3000)
X_mc_train = tfidf_mc.fit_transform(X_br_train)
X_mc_test  = tfidf_mc.transform(X_br_test)

# OvR dengan Logistic Regression
ovr_clf = OneVsRestClassifier(
    LogisticRegression(solver='liblinear', random_state=2024)
)
ovr_clf.fit(X_mc_train, y_br_train)
y_pred_ovr = ovr_clf.predict(X_mc_test)
acc_ovr = accuracy_score(y_br_test, y_pred_ovr)

# OvO dengan Logistic Regression
ovo_clf = OneVsOneClassifier(
    LogisticRegression(solver='liblinear', random_state=2024)
)
ovo_clf.fit(X_mc_train, y_br_train)
y_pred_ovo = ovo_clf.predict(X_mc_test)
acc_ovo = accuracy_score(y_br_test, y_pred_ovo)

# Native Logistic Regression (multinomial)
lr_native = LogisticRegression(
    solver='lbfgs', multi_class='multinomial',
    max_iter=1000, random_state=2024
)
lr_native.fit(X_mc_train, y_br_train)
y_pred_native = lr_native.predict(X_mc_test)
acc_native = accuracy_score(y_br_test, y_pred_native)

print("Perbandingan Strategi Multiclass (Brown Corpus):")
print(f"  One-vs-Rest (OvR):        {acc_ovr:.4f}")
print(f"  One-vs-One (OvO):         {acc_ovo:.4f}")
print(f"  Native Multinomial LR:    {acc_native:.4f}")
n_classifiers_ovo = len(unique_labels_br) * (len(unique_labels_br) - 1) // 2
print(f"
Jumlah kelas: {len(unique_labels_br)}")
print(f"Jumlah classifier OvO: {n_classifiers_ovo}")
print(f"Jumlah classifier OvR: {len(unique_labels_br)}")


In [ ]:
# Visualisasi perbandingan strategi
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart perbandingan accuracy
strategies = ['OvR', 'OvO', 'Native
Multinomial LR']
accs_mc    = [acc_ovr, acc_ovo, acc_native]
colors_mc  = ['steelblue', 'salmon', 'green']
bars = ax1.bar(strategies, accs_mc, color=colors_mc, edgecolor='black')
for bar, val in zip(bars, accs_mc):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11)
ax1.set_ylabel('Accuracy')
ax1.set_title('Perbandingan Strategi Multiclass
(Brown Corpus)')
ax1.set_ylim(0.5, 1.05)
ax1.grid(True, alpha=0.3, axis='y')

# Confusion matrix untuk strategi terbaik
best_mc_pred = y_pred_native if acc_native >= max(acc_ovr, acc_ovo) else                (y_pred_ovr if acc_ovr >= acc_ovo else y_pred_ovo)
cm_mc = pd.crosstab(
    pd.Series([unique_labels_br[l] for l in y_br_test], name='True'),
    pd.Series([unique_labels_br[l] for l in best_mc_pred], name='Predicted')
)
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Blues', ax=ax2, cbar=False,
            linewidths=0.5, annot_kws={'size': 7})
ax2.set_title('Confusion Matrix - Model Terbaik')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha='right', fontsize=8)
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle('Multiclass Classification Strategies - Brown Corpus', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6. Evaluasi Model Teks

### Penjelasan Teori

Evaluasi model teks menggunakan metrik yang sama dengan klasifikasi biasa, namun dengan beberapa pertimbangan khusus:

1. **Precision, Recall, F1 per kelas:** Sangat penting untuk memahami performa model di setiap kategori teks, terutama ketika distribusi kelas tidak seimbang.

2. **Macro vs Weighted average:** 
   - Macro: rata-rata sederhana per kelas (setiap kelas sama bobotnya).
   - Weighted: rata-rata tertimbang berdasarkan jumlah sampel per kelas.

3. **Confusion matrix:** Menunjukkan kelas mana yang sering tertukar, memberikan insight tentang hubungan semantik antar kelas.

**Evaluasi khusus model teks:**
- `zero_division=0`: Parameter penting untuk mencegah error pada kelas dengan zero predictions.
- **Perhatian pada kelas minoritas:** Kelas yang jarang dalam corpus cenderung memiliki performa rendah.

**Keterbatasan metrik ini untuk LLM:** Untuk Large Language Models yang menghasilkan teks (bukan mengklasifikasikannya), diperlukan metrik yang jauh lebih kompleks seperti BLEU, ROUGE, atau evaluasi manusia.


In [ ]:
# Evaluasi komprehensif pada movie_reviews (binary: pos vs neg)
from nltk.corpus import movie_reviews
import random

# Load seluruh movie_reviews corpus
docs_mr, labels_mr = [], []
for category in movie_reviews.categories():
    for fid in movie_reviews.fileids(category):
        docs_mr.append(" ".join(movie_reviews.words(fid)))
        labels_mr.append(category)

random.seed(2024)
combined = list(zip(docs_mr, labels_mr))
random.shuffle(combined)
docs_mr, labels_mr = zip(*combined)

X_mr_train, X_mr_test, y_mr_train, y_mr_test = train_test_split(
    list(docs_mr), list(labels_mr),
    test_size=0.3, random_state=2024
)

# TF-IDF vektorisasi
tfidf_mr = TfidfVectorizer(stop_words='english', max_features=5000)
X_mr_train_vect = tfidf_mr.fit_transform(X_mr_train)
X_mr_test_vect  = tfidf_mr.transform(X_mr_test)

# Logistic Regression
lr_mr = LogisticRegression(max_iter=1000, random_state=2024)
lr_mr.fit(X_mr_train_vect, y_mr_train)
y_mr_pred = lr_mr.predict(X_mr_test_vect)

from sklearn.metrics import precision_score, recall_score, f1_score

prec_mr = precision_score(y_mr_test, y_mr_pred, pos_label='pos')
rec_mr  = recall_score(y_mr_test, y_mr_pred, pos_label='pos')
f1_mr   = f1_score(y_mr_test, y_mr_pred, pos_label='pos')
acc_mr  = accuracy_score(y_mr_test, y_mr_pred)

print("Evaluasi Logistic Regression - Movie Reviews (Sentiment Analysis):")
print(f"  Accuracy:  {acc_mr:.4f}")
print(f"  Precision: {prec_mr:.4f}")
print(f"  Recall:    {rec_mr:.4f}")
print(f"  F1-Score:  {f1_mr:.4f}")


In [ ]:
# Visualisasi konfusi dan distribusi prediksi
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Confusion matrix
cm_mr = confusion_matrix(y_mr_test, y_mr_pred, labels=['neg', 'pos'])
disp  = ConfusionMatrixDisplay(confusion_matrix=cm_mr,
                                display_labels=['Negative', 'Positive'])
disp.plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title('Confusion Matrix
Sentiment Analysis (Movie Reviews)')

# Panel 2: Metrik per kelas
report_mr = classification_report(
    y_mr_test, y_mr_pred, output_dict=True, zero_division=0
)
classes   = ['neg', 'pos']
metrics_n = ['precision', 'recall', 'f1-score']
x_pos     = np.arange(len(metrics_n))
w         = 0.3

for i, cls in enumerate(classes):
    vals = [report_mr[cls][m] for m in metrics_n]
    ax2.bar(x_pos + i * w, vals, w,
            label=f'{"Negative" if cls=="neg" else "Positive"}',
            color='steelblue' if cls=='neg' else 'salmon',
            edgecolor='black')

ax2.set_xticks(x_pos + w / 2)
ax2.set_xticklabels(['Precision', 'Recall', 'F1-Score'])
ax2.set_ylabel('Score')
ax2.set_ylim(0, 1.15)
ax2.set_title('Metrik per Kelas
Sentiment Analysis (Movie Reviews)')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

for bars in ax2.containers:
    ax2.bar_label(bars, fmt='%.3f', padding=2, fontsize=9)

plt.suptitle('Evaluasi Komprehensif - Logistic Regression + TF-IDF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Analisis kata-kata paling berpengaruh (interpretasi model)
feature_names_mr = tfidf_mr.get_feature_names_out()
coef_mr = lr_mr.coef_[0]  # Koefisien untuk kelas 'pos'

top15_pos_idx = np.argsort(coef_mr)[-15:][::-1]  # Kata paling prediktif untuk 'pos'
top15_neg_idx = np.argsort(coef_mr)[:15]          # Kata paling prediktif untuk 'neg'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Kata-kata untuk sentimen positif
ax1.barh(feature_names_mr[top15_pos_idx][::-1],
         coef_mr[top15_pos_idx][::-1], color='salmon')
ax1.set_xlabel('Koefisien LR')
ax1.set_title('Top 15 Kata Prediktif untuk Sentimen POSITIF')
ax1.grid(True, alpha=0.3, axis='x')

# Kata-kata untuk sentimen negatif
ax2.barh(feature_names_mr[top15_neg_idx],
         np.abs(coef_mr[top15_neg_idx]), color='steelblue')
ax2.set_xlabel('|Koefisien LR|')
ax2.set_title('Top 15 Kata Prediktif untuk Sentimen NEGATIF')
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle('Interpretasi Model: Kata-kata Paling Berpengaruh
(Logistic Regression + TF-IDF)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretasi: Kata dengan koefisien besar (positif/negatif)")
print("adalah yang paling menentukan prediksi sentimen model.")


---
## Ringkasan Chapter 9

| Teknik | Kelas scikit-learn | Deskripsi |
|--------|-------------------|-----------|
| Bag-of-Words | `CountVectorizer()` | Hitungan kemunculan kata per dokumen |
| TF-IDF | `TfidfVectorizer()` | Bobot kata berdasarkan frekuensi dan keunikan |
| N-gram | `CountVectorizer(ngram_range=(1,2))` | Menangkap konteks urutan kata |
| POS Tagging | `nltk.pos_tag()` | Kategori gramatikal setiap kata |
| Naive Bayes | `MultinomialNB()` | Klasifikasi probabilistik, efisien untuk teks |
| SVM Teks | `SVC(kernel='linear')` | Linear kernel cocok untuk teks berdimensi tinggi |
| One-vs-Rest | `OneVsRestClassifier()` | Satu classifier binary per kelas |
| One-vs-One | `OneVsOneClassifier()` | Satu classifier per pasangan kelas |

### Poin Kunci untuk Diingat:

1. **Preprocessing teks adalah kunci:** Lowercase, stopword removal, dan vektorisasi yang tepat sangat menentukan performa model, sering lebih dari pilihan algoritma.

2. **TF-IDF umumnya lebih baik dari BoW:** TF-IDF memberi bobot lebih pada kata yang diskriminatif dan spesifik, mengurangi dominasi kata-kata yang sekedar sering muncul.

3. **N-gram menangkap konteks:** `ngram_range=(1,2)` sering meningkatkan performa karena bigram seperti "not good" atau "very bad" lebih informatif dari unigram individualnya.

4. **Selalu hapus stopwords:** Kata-kata seperti "the", "a", "is" tidak informatif untuk klasifikasi dan menambah noise ke model.

5. **SVM dengan linear kernel** sering menjadi pilihan terbaik untuk klasifikasi teks karena data teks secara natural berdimensi tinggi dan linear separable di ruang TF-IDF.

6. **Logistic Regression** memberikan interpretabilitas tambahan: koefisien model menunjukkan kata mana yang paling berpengaruh terhadap setiap kelas.

7. **Gunakan `zero_division=0`** dalam `classification_report()` untuk menghindari error pada kelas yang tidak diprediksi sama sekali.
